In [1]:
!pip install git+https://github.com/huggingface/transformers torch accelerate bitsandbytes langchain
!pip install evaluate
!pip install datasets
!pip install -U sentence-transformers
!pip install langchain_community
!pip install peft
!pip install rouge_score
!pip install bert_score
!pip install faiss-gpu

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-iij4gj4s
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-iij4gj4s
  Resolved https://github.com/huggingface/transformers to commit 52cb4034ada381fe1ffe8d428a1076e5411a8026
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.

In [2]:
from langchain.vectorstores import FAISS

In [3]:
!pip install langchain-experimental
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 14.5 MB/s eta 0:00:00


In [6]:
import pandas as pd
df = pd.read_csv("/content/covid_abstracts.csv")
df = df["abstract"]

In [4]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings

In [5]:
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
embedding_model = SentenceTransformerEmbeddings(model_name='BAAI/bge-large-zh-v1.5')

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:141: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/sett

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/30.4k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [7]:
text_splitter = SemanticChunker(
    embedding_model, breakpoint_threshold_type="percentile"
)
from langchain.docstore.document import Document
docs = [Document(page_content=entry) for entry in df]
docs = text_splitter.split_documents(docs)

In [8]:
len(df)

10000

In [9]:
faiss_db = FAISS.from_documents(docs, embedding_model)

In [10]:
import torch
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
)

In [11]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [12]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from peft import PeftModel
from langchain.retrievers import EnsembleRetriever

In [13]:
model_name = "filipealmeida/Mistral-7B-Instruct-v0.1-sharded"
base_model = AutoModelForCausalLM.from_pretrained(model_name,
                                             torch_dtype=torch.bfloat16,
                                             quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now set to True since model is quantized.


pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

pytorch_model-00001-of-00008.bin:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

pytorch_model-00002-of-00008.bin:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

pytorch_model-00003-of-00008.bin:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

pytorch_model-00004-of-00008.bin:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

pytorch_model-00005-of-00008.bin:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

pytorch_model-00006-of-00008.bin:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

ChunkedEncodingError: ('Connection broken: IncompleteRead(928130323 bytes read, 1018123549 more expected)', IncompleteRead(928130323 bytes read, 1018123549 more expected))

In [ ]:
!pip install rank_bm25

In [ ]:
from langchain.schema import Document
from langchain_community.retrievers import BM25Retriever
from typing import List

def custom_preprocessing_func(text: str) -> List[str]: # Change the input type to str
    return text.split()

bm25_retriever = BM25Retriever.from_texts(
    [doc.page_content for doc in docs],
    preprocess_func=custom_preprocessing_func
)
bm25_retriever.k = 2

In [ ]:
faiss_retriever = faiss_db.as_retriever(
    search_kwargs={'k': 2}
)

In [ ]:
peft_model_id = "ProElectro07/Projectxx2"
model = PeftModel.from_pretrained(base_model, peft_model_id)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from langchain.chains import SimpleSequentialChain, LLMChain

In [ ]:
##############
text_generation_pipeline = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=tokenizer,
    temperature=0.3,
    top_k=10,
    top_p=.85,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=500,
    num_return_sequences=1,
    # truncation=False,
    do_sample=True,
    # no_repeat_ngram_size=3,
    # early_stopping=True
)

text_generation_pipeline.model = model

mistral_llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

In [ ]:

from langchain.retrievers.multi_query import MultiQueryRetriever

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[.3, 0.7]
)

# retriever_from_llm = MultiQueryRetriever.from_llm(
#     retriever=ensemble_retriever, llm=mistral_llm
# )

In [ ]:
from bert_score import score
from evaluate import load
rouge = load('rouge')
bertscore = load('bertscore')
bleu = load('bleu')

In [ ]:
ds = pd.read_csv("/content/validate_dataset.csv")

In [ ]:
ds = ds["description"]

In [ ]:
predictions = []

In [ ]:
from langchain import PromptTemplate, LLMChain
from langchain.chains import SimpleSequentialChain
from langchain.chains import RetrievalQA

In [ ]:
prompt_template = ("""[INST]
You are an expert in evaluating the relevance of documents to a query.

Given the query and the 4 documents below:

Query: {question}
Documents: {context}

Task:
1. For each document, label it as "Relevant" if it helps to answer the query, or "Irrelevant" if it does not.

Example:
Relevant
Relevant
Irrelevant
Irrelevant

Please provide the labels only, with each of the 4 document's label on a new line like the above format.

[/INST]
""")



# Create a prompt instance
Prompt_Template = PromptTemplate.from_template(prompt_template)

# Create the RAG chain
RAG_chain = RetrievalQA.from_chain_type(
    mistral_llm,
    retriever=ensemble_retriever,
    chain_type_kwargs={"prompt": Prompt_Template}
)

predictions = []

for query in ds:
    response = RAG_chain({"query":query})
    print(response["result"])
    predictions.append(response["result"])

In [ ]:
predictions

In [ ]:
p = predictions

In [ ]:
responses = p

d = []

for response in responses:
  labels = response.strip().lower().split('\n')
  d.append(labels)

In [ ]:
for k in d:
  print(len(k))

4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4


In [ ]:
score = []

test = 0

for i in d:
  t = 0
  if i[0]=="relevant":
    t = t + .4
  if i[1]=="relevant":
    t = t + .3
  if i[2]=="relevant":
    t = t + .2
  if i[3]=="relevant":
    t = t + .1
  score.append(t)
  test = test + t

In [ ]:
(score)

[0.7,
 0.7,
 0.7,
 0.4,
 0.4,
 0.4,
 0.7,
 0.4,
 0.7,
 0.7,
 0.7,
 0.4,
 0.4,
 0.4,
 0.7,
 0.7,
 0.4,
 0.7,
 0.4,
 0.7,
 0.7,
 0.7,
 0.5,
 0.7,
 0.4,
 0.4,
 0.4,
 0.5,
 0.4,
 0.4,
 0.5,
 0.4,
 0.5,
 0.4,
 0.7,
 0.4,
 0.5,
 0.7,
 0.7,
 0.7,
 0.4,
 0.7,
 0.7,
 0.7,
 0.4,
 0.1,
 0.5,
 0.4,
 0.5,
 0.7]

In [ ]:
(test/50)*100

53.99999999999997